# `ngllib` remote client — notebook

Pairs with [`remote_server.ipynb`](./remote_server.ipynb). The server
should already be running (its cell 4 should have started a background
server thread).

This notebook walks through:

1. Connecting via `RemoteEnv` (handshake learns the real env's spaces).
2. `reset()` — first call lazy-launches Chromium on the server (~10–15 s).
3. A short step loop that pans + rotates the Neuroglancer camera, with
   per-step image display.
4. `close()` — disconnects (which also stops the server thread).

In [ ]:
import numpy as np
from IPython.display import display
from PIL import Image

from ngllib import RemoteEnv
from ngllib.distributed.transports import SocketTransport, FilesystemTransport

## 1. Connect to the server

Match the `HOST` / `PORT` from the server notebook. Use a generous
`timeout` so the first `reset()` (browser launch + Neuroglancer init)
doesn't trip the wire-recv deadline.

In [ ]:
HOST = "127.0.0.1"
PORT = 5555

env = RemoteEnv(SocketTransport.client(host=HOST, port=PORT, timeout=600))
print("connected")
print("observation_space:", env.observation_space)
print("action_space:    ", env.action_space)

# --- Filesystem alternative (paste the dirs the server printed) ---
# ACTION_DIR = "/tmp/ngllib_remote_XXXXXX/actions"
# OBS_DIR    = "/tmp/ngllib_remote_XXXXXX/obs"
# env = RemoteEnv(FilesystemTransport.client(
#     action_dir=ACTION_DIR, obs_dir=OBS_DIR, timeout=600,
# ))

## 2. Reset and inspect the initial observation

This is the slow call — Chromium launches on the server, Neuroglancer
initializes, the first screenshot is taken. Expect ~10–15 s on a fresh
server, ~1–2 s on subsequent resets.

In [ ]:
obs, info = env.reset(seed=0)
print("position (voxels):", obs["position"].tolist())
print("orientation:      ", obs["orientation"].round(3).tolist())
print("cross-section scale:", float(obs["xs_scale"][0]))
print("projection scale: ", float(obs["proj_scale"][0]))
print("image: shape={} dtype={}".format(obs["image"].shape, obs["image"].dtype))
display(Image.fromarray(obs["image"]))

## 3. Build an action

`action_type = 3` is **`edit_state`** — applies the position / scale /
orientation deltas without dispatching a mouse click. The other delta
fields are non-zero so each step visibly pans the camera in X and
rotates ~0.2 rad around the X axis.

(`action_type` 0/1/2 would be left/right/double click using `mouse_xy`
and `modifiers` instead, with the deltas ignored.)

In [ ]:
action = {
    "action_type":      3,
    "mouse_xy":         np.array([100, 100], dtype=np.float32),
    "modifiers":        np.array([0, 0, 0], dtype=np.int8),
    "delta_pos":        np.array([10, 0, 0], dtype=np.float32),
    "delta_xs_scale":   np.array([0], dtype=np.float32),
    "delta_orient":     np.array([0.2, 0, 0], dtype=np.float32),
    "delta_proj_scale": np.array([2000], dtype=np.float32),
}
assert env.action_space.contains(action)
print("action conforms to env.action_space")

## 4. Step a few times, display each resulting image

In [ ]:
N_STEPS = 5
frames = []
for i in range(N_STEPS):
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(obs["image"].copy())
    print(
        f"step {i+1}: pos={obs['position'].round(1).tolist()}, "
        f"reward={reward}, terminated={terminated}"
    )
    if terminated or truncated:
        break

print(f"\ndisplaying {len(frames)} frames:")
for i, img in enumerate(frames):
    print(f"frame {i+1}:")
    display(Image.fromarray(img))

## 5. Close

Sends `{"cmd": "close"}` to the server; the server's `serve()` loop
exits cleanly (the background thread in the server notebook terminates),
the transport is torn down on both ends, and the browser is closed.

In [ ]:
env.close()
print("closed")